# 16 · Minimal-Statistic Search

This notebook asks a sharper question than Notebook 15:

> What is the smallest local feature set that still preserves the zeta–GUE closeness signal against Poisson?

Rather than comparing broad feature families, we now test **single statistics** and **small combinations**.

## Candidate statistics

For normalized 5-gap windows, we test:

- entropy only
- reversal symmetry only
- unevenness only
- ratio mean only
- ratio std only
- two-feature combinations
- three-feature combinations

## Goal

For each candidate feature set, we measure whether:

\[
d(\text{zeta}, \text{Poisson}) > d(\text{zeta}, \text{GUE})
\]

using simple geometric distances after embedding.

This notebook is a minimal-sufficient-statistic search, not a theorem notebook.

In [ ]:
import itertools
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
# Zeta gaps
N = 1000
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# Finite GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=50, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=140, n_matrices=50, bulk_fraction=0.5, rng=rng)

len(zeta_gaps), len(poisson_gaps), len(gue_gaps)

## Window and scalar feature helpers

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

feature_map = {
    "entropy": window_entropy,
    "symmetry": reversal_symmetry_score,
    "unevenness": unevenness,
    "ratio_mean": ratio_mean,
    "ratio_std": ratio_std,
}

## Build descriptor matrices from selected feature names

In [ ]:
def build_feature_matrix(gaps, feature_names, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [feature_map[name](w) for name in feature_names]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## Embedding and distance helpers

In [ ]:
def pca_embed_three(A, B, C):
    all_desc = np.vstack([A, B, C])
    mean = all_desc.mean(axis=0)
    std = all_desc.std(axis=0)
    std[std == 0] = 1.0

    A_std = (A - mean) / std
    B_std = (B - mean) / std
    C_std = (C - mean) / std

    X = np.vstack([A_std, B_std, C_std])
    Xc = X - X.mean(axis=0)

    if Xc.shape[1] == 1:
        emb = np.column_stack([Xc[:, 0], np.zeros(len(Xc))])
    else:
        cov = np.cov(Xc, rowvar=False)
        eigvals, eigvecs = np.linalg.eigh(cov)
        order = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, order]
        comps = eigvecs[:, :2]
        emb = Xc @ comps

    nA, nB, nC = len(A), len(B), len(C)
    return emb[:nA], emb[nA:nA+nB], emb[nA+nB:]

def centroid_distance(A, B):
    return float(np.linalg.norm(A.mean(axis=0) - B.mean(axis=0)))

def density_grid(X, bins_x, bins_y):
    H, _, _ = np.histogram2d(X[:, 0], X[:, 1], bins=[bins_x, bins_y], density=True)
    return H

def grid_l2_distance(H1, H2):
    return float(np.linalg.norm(H1.ravel() - H2.ravel()))

def wasserstein_1d(a, b):
    a = np.sort(np.asarray(a))
    b = np.sort(np.asarray(b))
    n = min(len(a), len(b))
    if len(a) != n:
        idx = np.linspace(0, len(a) - 1, n).astype(int)
        a = a[idx]
    if len(b) != n:
        idx = np.linspace(0, len(b) - 1, n).astype(int)
        b = b[idx]
    return float(np.mean(np.abs(a - b)))

def compute_feature_set_metrics(zeta_X, poisson_X, gue_X):
    zeta_emb, poisson_emb, gue_emb = pca_embed_three(zeta_X, poisson_X, gue_X)

    xmin = min(zeta_emb[:,0].min(), poisson_emb[:,0].min(), gue_emb[:,0].min())
    xmax = max(zeta_emb[:,0].max(), poisson_emb[:,0].max(), gue_emb[:,0].max())
    ymin = min(zeta_emb[:,1].min(), poisson_emb[:,1].min(), gue_emb[:,1].min())
    ymax = max(zeta_emb[:,1].max(), poisson_emb[:,1].max(), gue_emb[:,1].max())

    if xmin == xmax:
        xmin, xmax = xmin - 1, xmax + 1
    if ymin == ymax:
        ymin, ymax = ymin - 1, ymax + 1

    bins_x = np.linspace(xmin, xmax, 24)
    bins_y = np.linspace(ymin, ymax, 24)

    zeta_H = density_grid(zeta_emb, bins_x, bins_y)
    poisson_H = density_grid(poisson_emb, bins_x, bins_y)
    gue_H = density_grid(gue_emb, bins_x, bins_y)

    cd_zg = centroid_distance(zeta_emb, gue_emb)
    cd_zp = centroid_distance(zeta_emb, poisson_emb)

    gd_zg = grid_l2_distance(zeta_H, gue_H)
    gd_zp = grid_l2_distance(zeta_H, poisson_H)

    w1_zg = wasserstein_1d(zeta_emb[:,0], gue_emb[:,0])
    w1_zp = wasserstein_1d(zeta_emb[:,0], poisson_emb[:,0])

    w2_zg = wasserstein_1d(zeta_emb[:,1], gue_emb[:,1])
    w2_zp = wasserstein_1d(zeta_emb[:,1], poisson_emb[:,1])

    return {
        "centroid_gap": cd_zp - cd_zg,
        "grid_gap": gd_zp - gd_zg,
        "wasserstein_pc1_gap": w1_zp - w1_zg,
        "wasserstein_pc2_gap": w2_zp - w2_zg,
        "zeta_emb": zeta_emb,
        "poisson_emb": poisson_emb,
        "gue_emb": gue_emb,
    }

## Evaluate single, pair, and triple feature sets

In [ ]:
feature_names = list(feature_map.keys())

candidate_sets = []
for r in [1, 2, 3]:
    candidate_sets.extend(list(itertools.combinations(feature_names, r)))

zeta_base = zeta_gaps[:700]
poisson_base = poisson_gaps[:700]
gue_base = gue_gaps[:950]

results = []

for feat_set in candidate_sets:
    _, zeta_X = build_feature_matrix(zeta_base, feat_set, k=5)
    _, poisson_X = build_feature_matrix(poisson_base, feat_set, k=5)
    _, gue_X = build_feature_matrix(gue_base, feat_set, k=5)

    m = min(len(zeta_X), len(poisson_X), len(gue_X))
    metrics = compute_feature_set_metrics(zeta_X[:m], poisson_X[:m], gue_X[:m])

    results.append({
        "feature_set": feat_set,
        "n_features": len(feat_set),
        "centroid_gap": metrics["centroid_gap"],
        "grid_gap": metrics["grid_gap"],
        "wasserstein_pc1_gap": metrics["wasserstein_pc1_gap"],
        "wasserstein_pc2_gap": metrics["wasserstein_pc2_gap"],
        "zeta_emb": metrics["zeta_emb"],
        "poisson_emb": metrics["poisson_emb"],
        "gue_emb": metrics["gue_emb"],
    })

len(results)

## Sort by centroid-gap strength

In [ ]:
sorted_results = sorted(results, key=lambda r: r["centroid_gap"], reverse=True)

top10 = [
    {
        "feature_set": r["feature_set"],
        "n_features": r["n_features"],
        "centroid_gap": r["centroid_gap"],
        "grid_gap": r["grid_gap"],
        "wasserstein_pc1_gap": r["wasserstein_pc1_gap"],
        "wasserstein_pc2_gap": r["wasserstein_pc2_gap"],
    }
    for r in sorted_results[:10]
]

top10

## Best single-feature, two-feature, and three-feature sets

In [ ]:
best_by_size = {}
for size in [1, 2, 3]:
    candidates = [r for r in sorted_results if r["n_features"] == size]
    best_by_size[size] = {
        "feature_set": candidates[0]["feature_set"],
        "centroid_gap": candidates[0]["centroid_gap"],
        "grid_gap": candidates[0]["grid_gap"],
        "wasserstein_pc1_gap": candidates[0]["wasserstein_pc1_gap"],
        "wasserstein_pc2_gap": candidates[0]["wasserstein_pc2_gap"],
    }

best_by_size

## Gap distributions by feature-set size

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=False)

for ax, size in zip(axes, [1, 2, 3]):
    vals = [r["centroid_gap"] for r in results if r["n_features"] == size]
    ax.hist(vals, bins=10)
    ax.set_title(f"{size}-feature sets")
    ax.set_xlabel("centroid gap")
    ax.set_ylabel("count")

plt.tight_layout()
plt.show()

## Top centroid-gap feature sets

In [ ]:
labels = [" + ".join(r["feature_set"]) for r in sorted_results[:12]]
vals = [r["centroid_gap"] for r in sorted_results[:12]]

plt.figure(figsize=(12, 4.8))
plt.bar(np.arange(len(labels)), vals)
plt.xticks(np.arange(len(labels)), labels, rotation=35, ha="right")
plt.ylabel("centroid gap")
plt.title("Top feature sets by centroid gap")
plt.tight_layout()
plt.show()

## Representative embeddings

Show the best:
- 1-feature set
- 2-feature set
- 3-feature set

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

for ax, size in zip(axes, [1, 2, 3]):
    best = next(r for r in sorted_results if r["n_features"] == size)
    ax.scatter(best["zeta_emb"][:,0], best["zeta_emb"][:,1], s=8, alpha=0.22, label="zeta")
    ax.scatter(best["poisson_emb"][:,0], best["poisson_emb"][:,1], s=8, alpha=0.22, label="Poisson")
    ax.scatter(best["gue_emb"][:,0], best["gue_emb"][:,1], s=8, alpha=0.22, label="GUE")
    ax.set_title(" + ".join(best["feature_set"]))
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

axes[0].legend()
plt.tight_layout()
plt.show()

## Minimality summary

We check whether the best 1-feature, 2-feature, and 3-feature sets all preserve the basic inequality:
\[
d(\text{zeta}, \text{Poisson}) > d(\text{zeta}, \text{GUE})
\]
across multiple metrics.

In [ ]:
minimality_summary = {
    f"best_{size}_feature_set": best_by_size[size]
    for size in [1, 2, 3]
}
minimality_summary

## Final summary

In [ ]:
summary = {
    "top10": top10,
    "best_by_size": best_by_size,
    "n_total_feature_sets": len(results),
}
summary

## Notes

- If a single feature already gives a positive and substantial gap, that suggests the universality signal is highly concentrated.
- If only two- or three-feature sets work well, that suggests the signal is distributed but still low-dimensional.
- Comparing the best sets by size helps identify plausible minimal sufficient statistics for the local zeta–GUE correspondence.

## Next directions

A natural next notebook could do one of these:

1. bootstrap the best single- and two-feature sets for uncertainty intervals  
2. run the same minimal-statistic search on conditional windows (after small vs after large gaps)  
3. vary the window length and repeat the search  
4. compare minimal-statistic behavior with classifier accuracy